# 🧬 LifeOps: GRPO Training — Chaotic Life Management

Trains `Qwen2.5-3B-Instruct` with GRPO to balance Career, Family, and Health.

Run this notebook on **Colab T4 (free tier)**.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade "trl>=0.15.0" transformers datasets httpx matplotlib openenv-core python-dotenv mergekit

## 2. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print("✅ Model loaded")

## 3. Launch LifeOps Server (Robust Background)

In [ ]:
import os
import subprocess
import time
import httpx
import sys

# 1. Extract and Navigate
if os.path.exists("LifeOps.zip"):
    !unzip -o LifeOps.zip
    # Check if unzipped into a subfolder
    if os.path.exists("LifeOps") and os.path.isdir("LifeOps"):
        %cd LifeOps

# 2. Launch Server with Logging
print("🚀 Launching LifeOps Server...")
with open("server_log.txt", "w") as log:
    subprocess.Popen(["python", "-m", "server.app"], 
                     stdout=log, stderr=log, 
                     env=dict(os.environ, PYTHONPATH=os.getcwd()))

# 3. Retry loop for connection
ENV_URL = "http://localhost:8000"
connected = False
for i in range(10):
    time.sleep(3)
    try:
        with httpx.Client(base_url=ENV_URL, timeout=5) as http:
            health = http.get("/health")
            if health.status_code == 200:
                print(f"✅ LifeOps Server is LIVE at {ENV_URL}")
                connected = True
                break
    except:
        print(f"Attempt {i+1}: Waiting for server...")

if not connected:
    print("❌ Server failed. LOG OUTPUT:")
    if os.path.exists("server_log.txt"):
        with open("server_log.txt", "r") as f: print(f.read())

## 4. Define Reward Function

In [ ]:
import re

# Debug window for inspecting raw completions during training.
DEBUG_START_STEP = 3
DEBUG_END_STEP = 8
_reward_call_step = 0


def _extract_action_name(text: str) -> str | None:
    match = re.search(r"Action\s*:\s*([a-zA-Z_ -]+)", text, re.IGNORECASE)
    if not match:
        return None
    return re.sub(r"[\s-]+", "_", match.group(1).strip().lower())


def _format_adjustment(text: str) -> float:
    """
    Small shaping term to encourage strict 2-line output:
    Action + Justification, and no extra rambling afterwards.
    """
    score = 0.0
    has_action = re.search(r"^\s*Action\s*:", text, re.IGNORECASE | re.MULTILINE) is not None
    just_match = re.search(r"^\s*Justification\s*:\s*(.+)", text, re.IGNORECASE | re.MULTILINE)

    if has_action:
        score += 0.2
    else:
        score -= 0.5

    if just_match:
        score += 0.2
    else:
        score -= 0.5

    # Penalize non-empty text after the Justification line.
    if just_match:
        after = text[just_match.end():]
        if after.strip():
            score -= 0.6

    # Mild length penalty for long rambling completions.
    if len(text) > 220:
        score -= 0.2

    return score


def reward_fn(completions: list[str], **kwargs) -> list[float]:
    global _reward_call_step
    _reward_call_step += 1

    rewards = []
    with httpx.Client(base_url=ENV_URL, timeout=10) as http:
        for idx, completion in enumerate(completions):
            action_str = _extract_action_name(completion)

            if not action_str:
                total_reward = -1.0 + _format_adjustment(completion)
                rewards.append(total_reward)
                continue

            try:
                http.post("/reset")
                resp = http.post("/step", json={"action": {"choice": action_str, "justification": "RL Step"}})
                env_reward = float(resp.json().get("reward", 0.0)) if resp.status_code == 200 else -2.0
            except Exception:
                env_reward = 0.0

            total_reward = env_reward + _format_adjustment(completion)
            rewards.append(total_reward)

    # Print raw completions around target steps for diagnosis.
    if DEBUG_START_STEP <= _reward_call_step <= DEBUG_END_STEP:
        print(f"\n===== DEBUG STEP {_reward_call_step} =====")
        for i, completion in enumerate(completions[:5]):
            action = _extract_action_name(completion)
            snippet = completion[:220].replace("\n", "\\n")
            print(
                f"sample={i} len={len(completion)} action={action} "
                f"format_adj={_format_adjustment(completion):+.2f} "
                f"reward={rewards[i]:+.2f}"
            )
            print(f"text: {snippet}")

    return rewards

## 5. Prepare Dataset

In [ ]:
from datasets import Dataset
import sys
import inspect

sys.path.append(os.getcwd())
from scripts.dataset_builder import LifeopsDatasetBuilder


def _truncate_prompt_text(text: str, max_chars: int = 1200) -> str:
    """Keep prompts bounded when TRL no longer exposes GRPOConfig.max_prompt_length."""
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 20] + "\n\n[TRUNCATED]"


raw_items = LifeopsDatasetBuilder.generate_rl_dataset(100)
rows = []
for item in raw_items:
    prompt = f"{item['prompt'][0]['content']}\n\n{item['prompt'][1]['content']}"
    rows.append({"prompt": _truncate_prompt_text(prompt)})

dataset = Dataset.from_list(rows)
print(f"✅ Generated {len(dataset)} examples")

try:
    from trl import GRPOConfig

    sig = inspect.signature(GRPOConfig.__init__)
    print("TRL GRPOConfig supports max_prompt_length:", "max_prompt_length" in sig.parameters)
    print("TRL GRPOConfig supports max_completion_length:", "max_completion_length" in sig.parameters)
except Exception as e:
    print("Could not introspect GRPOConfig:", repr(e))

## 6. GRPO Training

In [ ]:
import inspect
from trl import GRPOConfig, GRPOTrainer

# Newer Unsloth versions already patch GRPO internals automatically.
# On Colab / Python 3.12, explicit PatchFastRL() can raise OSError.
try:
    from unsloth import PatchFastRL
    try:
        PatchFastRL()
    except OSError:
        pass
except Exception:
    pass


def _build_grpo_config(**kwargs) -> GRPOConfig:
    """Build GRPOConfig across TRL versions (some drop max_prompt_length)."""
    sig = inspect.signature(GRPOConfig.__init__)
    filtered = {k: v for k, v in kwargs.items() if k in sig.parameters}
    dropped = sorted(set(kwargs) - set(filtered))
    if dropped:
        print("Note: dropping unsupported GRPOConfig args:", dropped)
    return GRPOConfig(**filtered)


cfg = _build_grpo_config(
    num_generations=4,
    max_steps=100,
    learning_rate=2e-5,
    max_prompt_length=384,  # only used on older TRL; ignored otherwise
    max_completion_length=96,
    temperature=0.9,
    output_dir="./grpo_out",
    logging_steps=1,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=cfg,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("GRPO config loaded:")
if hasattr(trainer.args, "max_prompt_length"):
    print("- max_prompt_length:", trainer.args.max_prompt_length)
else:
    print("- max_prompt_length: (not supported by this TRL version; prompts truncated in dataset cell)")
print("- max_completion_length:", getattr(trainer.args, "max_completion_length", None))
print("- num_generations:", trainer.args.num_generations)
print("- temperature:", getattr(trainer.args, "temperature", None))

gen_cfg = getattr(trainer.model, "generation_config", None)
if gen_cfg is not None:
    print("Model generation_config:")
    print("- max_new_tokens:", getattr(gen_cfg, "max_new_tokens", None))
    print("- eos_token_id:", getattr(gen_cfg, "eos_token_id", None))

trainer.train()